# 혜택(benefit) 축 선정 — 무엇을 넣고 무엇을 뺐나

`cost·data·qos·tether·voice·sms·ott` 7축 중 혜택에서 나온 건 `ott_value` 하나뿐이다.
혜택 데이터가 자유텍스트라 다양해 보이지만, 실제로 뭘 넣고 뺄지는 아래 6개 기준으로
판단한다.

1. 커버리지 — 이 혜택이 요금제 몇 %에 있나. 너무 적으면 축으로 못 쓴다.
2. 반복가치 vs 1회성 — 매달 받는 혜택인가, 가입 시 한 번 받는 쿠폰/캐시백인가.
3. 금액 환산 가능 여부 — 공신력 있는 가격표가 있는가, 없으면 숫자를 지어내는 꼴이다.
4. 텍스트 정규화 필요 여부 — 같은 혜택이 문구만 달라 여러 개로 잡히는 건 아닌가.
5. 택1 vs 다 제공 — 한 플랜에 여러 혜택이 뜨면 다 받는 건지 하나만 고르는 건지.
6. 이미 있는 축과 중복인가 — 이중계산 방지.

이 노트북은 이 기준을 실제 데이터에 적용해서 확인한 기록이다.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

DATA = Path("../data")
plans = pd.read_csv(DATA / "통신요금제_통합데이터_최종.csv", dtype={"plan_id": str})
benefits = pd.read_csv(DATA / "통신요금제_혜택상세_최종.csv", dtype={"plan_id": str})
prices = pd.read_csv(DATA / "ott_prices.csv")
price_map = dict(zip(prices.service, prices.monthly_won.astype(int)))

# 학습 모집단: MVNO 중 SKT/KT/LG U+ 로 찍힌 45개(온라인전용 오분류) 제외
mvno_all = plans[plans.carrier_type == "MVNO"].copy()
MNO_BRANDS = ["SKT", "KT", "LG U+", "LGU+"]
mvno_ids = set(mvno_all[~mvno_all.mvno_brand.isin(MNO_BRANDS)].plan_id)
bm = benefits[benefits.plan_id.isin(mvno_ids)]

print(f"학습 모집단 {len(mvno_ids):,}개 · 혜택 행 {len(bm):,}개")

학습 모집단 2,158개 · 혜택 행 2,511개


## 1. 카테고리 개수부터 확인

"혜택이 다양하다"는 인상은 카테고리 개수가 아니라 카테고리 안 텍스트가 제각각이라
생기는 것이다. 실제 `benefit_category`가 몇 종류인지부터 본다.

In [2]:
plan_n = len(mvno_ids)
summary = bm.groupby("benefit_category").agg(
    플랜수=("plan_id", "nunique"),
    평균시장가원=("benefit_value_won", "mean"),
    시장가결측률=("benefit_value_won", lambda s: s.isna().mean()),
)
summary["커버리지(%)"] = (summary["플랜수"] / plan_n * 100).round(1)
summary.round(2).sort_values("커버리지(%)", ascending=False)

,플랜수,평균시장가원,시장가결측률,커버리지(%)
benefit_category,,,,
사은품/페이백,1080,26392.35,0.33,50.0
추가데이터,322,NaN,1.00,14.9
OTT/구독,253,11900.00,0.94,11.7


딱 3개다 — 사은품/페이백(50.0%), 추가데이터(14.9%), OTT/구독(11.7%). 이 3개를
기준 1~6에 하나씩 통과시킨다.

## 2. 추가데이터 — 이미 다른 축에 흡수됨

반복가치 있고 GB로 환산 가능하지만(기준 2·3 통과), `data_total_gb` 계산에 이미
합쳐져 있다(기준 6에서 걸림) — 별도 축 불필요.

In [3]:
extra = bm[bm.benefit_category == "추가데이터"].copy()
extra["gb"] = extra.benefit_name.fillna("").str.extract(r"([\d.]+)\s*GB", flags=re.I)[0].astype(float)
print(f"추가데이터 있는 요금제 {extra.plan_id.nunique()}개, GB 파싱 실패(NaN) {extra.gb.isna().sum()}개")
extra[["plan_id", "benefit_name", "gb"]].head()

추가데이터 있는 요금제 322개, GB 파싱 실패(NaN) 2개


,plan_id,benefit_name,gb
2913,26613,SOLO결합(+20GB),20.0
2914,26613,데이터쿠폰 20GB,20.0
2922,37103,KT 데이터 결합 (추가데이터 20GB),20.0
2941,6830,"헬로모바일 결합시, 추가 데이터 5GB 증정",5.0
2950,30954,KT 데이터 결합 (추가데이터 20GB),20.0


## 3. OTT/구독 — 반복가치 있고 가격표로 환산 가능

기준 3(금액 환산)을 유일하게 통과하는 카테고리. 근데 카테고리 커버리지(11.7%)를
그대로 믿으면 안 된다 — 서비스명이 없는 행이 섞여 있어서 실제 영상 OTT 커버리지는
이보다 낮다.

In [4]:
ott = bm[bm.benefit_category == "OTT/구독"]
unmapped = ott[ott.benefit_service.isna()]
valid = ott[ott.benefit_service.notna()]

print(f"OTT 카테고리 커버리지: {ott.plan_id.nunique()}개 ({ott.plan_id.nunique()/plan_n:.1%})")
print(f"서비스명 없는 행(영상 아닌 구독): {len(unmapped)}개 ({len(unmapped)/len(ott):.1%})")
print(f"실제 영상 OTT 커버리지: {valid.plan_id.nunique()}개 ({valid.plan_id.nunique()/plan_n:.1%})")
print()
print("서비스명 없는 행 샘플(뭐가 섞여있나):")
unmapped.benefit_name.value_counts().head(5)

OTT 카테고리 커버리지: 253개 (11.7%)
서비스명 없는 행(영상 아닌 구독): 213개 (69.8%)
실제 영상 OTT 커버리지: 92개 (4.3%)

서비스명 없는 행 샘플(뭐가 섞여있나):


benefit_name
조선일보 종이신문 1개월 무료 구독 이벤트    87
구글 AI프로+도미노피자 할인쿠폰         51
토스미·오픽미 AI 모의고사            51
예스24 크레마클럽 구독권 (24개월)       7
모아진(전세계 매거진) 무료 구독          6
Name: count, dtype: int64

## 4. 사은품/페이백 — 커버리지 50%로 제일 높은데 왜 뺐나

기준 1(커버리지)만 보면 셋 중 제일 좋은 후보다. 근데 기준 2(반복가치)에서 걸린다 —
가입 시 한 번 받는 쿠폰·캐시백·유심비 면제라 "이 요금제를 계속 쓸 때의 가치"를
반영하지 못한다. 텍스트를 정규화해도(기준 4) 이 판단은 안 바뀐다 — 유심비 면제도
표기만 6가지로 쪼개져 있을 뿐 결국 1회성이다.

게다가 같은 혜택이 채널별로 문구만 바꿔 중복 나열되는 문제도 있다(기준 3 — 금액을
못 믿음). 실제 사례로 확인한다.

In [5]:
gift = bm[bm.benefit_category == "사은품/페이백"]
print(f"사은품/페이백 {len(gift)}행, benefit_name 유니크 {gift.benefit_name.nunique()}개")
print()
print("표기만 다르고 사실상 같은 혜택(유심/배송비 무료류) 예시:")
sim_like = gift[gift.benefit_name.str.contains("유심", na=False)]
sim_like.benefit_name.value_counts()

사은품/페이백 1793행, benefit_name 유니크 145개

표기만 다르고 사실상 같은 혜택(유심/배송비 무료류) 예시:


benefit_name
KT유심&배송비 무료              78
일반유심/배송비 무료              78
일반 유심/배송비 무료             54
개통완료 시 유심/배송비 무료 1개      38
유심/배송비 무료                36
NFC 유심 무료 제공             36
유심/배송비 무료!               32
쿠팡유심 개통 시 쿠팡캐시 2만원       25
유심사 3일 500MB 할인 쿠폰 지급     4
유심비 지원(신세계상품권 1만원)        2
Name: count, dtype: int64

## 5. 택1 vs 다 제공 — 기준 5, 아직 안 풀었던 것

`is_selectable` 컬럼은 못 믿는다 — MVNO 혜택 전부 False로 찍혀 있다. 그래서 실제로
"A 또는 B 중 1개" 같은 문구가 데이터에 얼마나 있는지, 있다면 학습 모집단에도
영향을 주는지 텍스트로 직접 확인한다.

In [6]:
PICK_ONE_PATTERNS = ["택1", "중 1개", "중1개", "1개 선택", "1종 선택", "중 하나", "중에서", "택 1", "1개만"]
has_marker = benefits.benefit_name.fillna("").str.contains("|".join(PICK_ONE_PATTERNS))
hit = benefits[has_marker].copy()
hit["in_train_pop"] = hit.plan_id.isin(mvno_ids)

print(f"전체 혜택 {len(benefits):,}행 중 택1 문구 포함: {len(hit)}개")
print(f"그중 학습 모집단 안에 있는 것: {hit.in_train_pop.sum()}개")
print()
hit[hit.in_train_pop][["plan_id", "benefit_category", "benefit_name"]]

전체 혜택 5,581행 중 택1 문구 포함: 23개
그중 학습 모집단 안에 있는 것: 2개



,plan_id,benefit_category,benefit_name
3409,35523,사은품/페이백,편의점 상품권(택 1) 매월 제공
3992,35524,사은품/페이백,캐시 상품권(택 1) 매월 제공


학습 모집단 안에 있는 건 2개뿐이고, 둘 다 사은품/페이백(위에서 이미 통째로 제외한
카테고리) — 영향 없음. OTT 쪽 택1 문구(T우주패스/FLO 중 1개 등)는 전부 오분류
45개(온라인전용) 안에 있어서 학습 모집단 밖이다.

명시적 문구는 이걸로 확인됐는데, 더 중요한 건 **문구 없이 조용히 여러 개 나열된
경우**다 — 이게 진짜 위험한 케이스다. 학습 모집단 안에서 서비스명이 서로 다른 OTT를
2개 이상 가진 요금제가 있는지 직접 센다.

In [7]:
multi = valid.groupby("plan_id")["benefit_service"].apply(lambda s: sorted(set(s)))
multi_2plus = multi[multi.apply(len) >= 2]

print(f"학습 모집단 OTT 유효행 {len(valid)}개 / 서로 다른 서비스 2개 이상 가진 plan_id: {len(multi_2plus)}개")
multi_2plus

학습 모집단 OTT 유효행 92개 / 서로 다른 서비스 2개 이상 가진 plan_id: 0개


Series([], Name: benefit_service, dtype: object)

## 6. 정리

0개다. 학습 모집단 안에서는 OTT 있는 요금제가 전부 서비스 1개씩만 가진다 — 택1/다
제공을 구분해야 하는 상황 자체가 안 생긴다. `scoring.py`의 `set()` 방어 로직은
실제로는 한 번도 발동한 적 없는 셈이다.

| 카테고리 | 커버리지 | 채택 여부 | 이유 |
|---|---|---|---|
| 추가데이터 | 14.9% | 별도 축 아님 | `data_total_gb`에 이미 흡수(기준 6) |
| OTT/구독 | 11.7%(실제 영상 5.4%) | `ott_value` 채택 | 반복가치+가격표 있음(기준 2·3), 택1 문제 없음(기준 5) |
| 사은품/페이백 | 50.0% | 제외 | 1회성(기준 2), 채널별 중복 나열(기준 3) |

택1/다제공 애매함은 학습 모집단이 아니라 오분류 45개·통신 3사 쪽에서만 나타난다 —
이미 다른 이유(브랜드 기준)로 걸러진 영역과 겹친다. 서빙 모집단이 넓어지거나(예:
통신 3사까지 같은 로직으로 스코어링) 새 크롤링에서 다중 OTT 조합이 생기면 이 결론은
다시 확인해야 한다.